# Room twin: 3D workstation + 2-min fly-through + web viewer

What this builds:
- From two seated clips of the same workstation (`MTL-SKL00089-006`, `MTL-SKL00089-007`), an accurate, continuous
  **3D copy of the workstation**.
- The worker's **hand meshes, head and gaze** from the first 2 min of `007`, replayed inside it.
- `room_video.mp4`: a 2-min fly-through.
- `room_web.zip`: a viewer you open on your computer. Orbit around the room and drag a time slider to move the
  hands forward and back.

**Before you start**
1. Run the PS4 notebook once with `MESH_FOR = ["MTL-SKL00089-006", "MTL-SKL00089-007"]`. This adds
   `hands_mesh.npz` to their PS4 result folders. Without it, the hands are drawn as skeletons.
2. Put `ps5_slam.zip` (code v1.4 or newer) in `MyDrive/mytronD/`.
3. Runtime → **A100**, fresh runtime → **Run all**.

About 1 h in total: install 15-20 min, SLAM 2 x ~7 min, room ~15 min, video ~10-15 min.

Results go to `MyDrive/mytronD/room_twin/MTL-SKL00089-007/`. *The outputs are made from the footage: keep them
private and delete them after the event.*

In [ ]:
# CELL 1: install ViPE (compiles CUDA kernels, ~10-20 min), MoGe-2 depth, Open3D
import torch, os
assert torch.cuda.is_available(), "switch the runtime to a GPU (A100)"
cc = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"
print(torch.cuda.get_device_name(0), "| arch", os.environ["TORCH_CUDA_ARCH_LIST"], "| torch", torch.__version__)
!pip -q install ninja setuptools wheel
!pip -q install --no-build-isolation nvidia-vipe
!pip -q install rerun-sdk pandas pyyaml scipy opencv-python git+https://github.com/cvg/GeoCalib
!pip -q install open3d trimesh click
!pip -q install --no-deps git+https://github.com/microsoft/MoGe.git "git+https://github.com/EasternJournalist/utils3d-moge.git@62f09d58509485564e24d5d9f6aac9ee9ebc0c37"
import vipe, open3d, trimesh
from moge.model.v2 import MoGeModel
print("ViPE, Open3D", open3d.__version__, ", MoGe: installed")

In [ ]:
# CELL 2: code from Drive (newest version), self-tests (~2 min)
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil, os, glob, re, sys
DRIVE_DIR = "/content/drive/MyDrive/mytronD"
NEED_WORLD, NEED_ROOM = (1, 4), (1, 0)

def versions(z):
    try:
        zf = zipfile.ZipFile(z); names = zf.namelist()
        def ver(suffix):
            f = [n for n in names if n.endswith(suffix)]
            m = re.search(r'VERSION = "(\d+)\.(\d+)"', zf.read(f[0]).decode()) if f else None
            return (int(m.group(1)), int(m.group(2))) if m else None
        return ver("ps5world/__init__.py"), ver("room/__init__.py")
    except zipfile.BadZipFile:
        return None, None

zips = {z: versions(z) for z in glob.glob(f"{DRIVE_DIR}/*.zip")}
for z, v in zips.items():
    print(f"  {os.path.basename(z):40s} world {v[0]} room {v[1]}")
good = sorted([z for z, v in zips.items() if v[0] and v[1] and v[0] >= NEED_WORLD and v[1] >= NEED_ROOM],
              key=lambda z: (zips[z], os.path.getmtime(z)), reverse=True)
assert good, f"no zip with world >= {NEED_WORLD} and room >= {NEED_ROOM}: upload the new ps5_slam.zip to {DRIVE_DIR}"
ZIP = good[0]
shutil.rmtree("/content/_src", ignore_errors=True); shutil.rmtree("/content/code", ignore_errors=True)
zipfile.ZipFile(ZIP).extractall("/content/_src")
root = os.path.dirname(sorted(glob.glob("/content/_src/**/run_room.py", recursive=True), key=len)[0])
shutil.move(root, "/content/code")
print("code from", ZIP, zips[ZIP])
!cd /content/code && python tests/test_world_synthetic.py 2>&1 | tail -1
!cd /content/code && python tests/test_room_synthetic.py 2>&1 | tail -1

In [ ]:
# CELL 3: settings, find the inputs, copy the videos locally
CLIPS    = ["MTL-SKL00089-006", "MTL-SKL00089-007"]   # same workstation; both are used for the 3D room
TARGET   = "MTL-SKL00089-007"                          # its hands / head are replayed
SECONDS  = 120                                         # length of the video and the web motion
CLIPS_DIR = DRIVE_DIR                                  # folder with the videos ("Copy of ..." is fine)
OUT_DIR  = f"{DRIVE_DIR}/room_twin/{TARGET}"
import os, glob, shutil
def clean(p): return os.path.splitext(os.path.basename(p))[0].replace("Copy of ", "").strip().replace(" ", "_")

def find_ps4(name):                                    # any results folder in My Drive (ps4v5_FINAL, ps4v4_results, ...)
    hits = glob.glob(f"/content/drive/MyDrive/*/{name}/hands.jsonl") + glob.glob(f"/content/drive/MyDrive/*/*/{name}/hands.jsonl")
    hits = sorted(hits, key=lambda p: (not os.path.exists(os.path.join(os.path.dirname(p), "hands_mesh.npz")), len(p)))
    return os.path.dirname(hits[0]) if hits else None

LOCAL_CLIPS, PS4_LINKS = "/content/clips", "/content/ps4_links"
os.makedirs(LOCAL_CLIPS, exist_ok=True); os.makedirs(PS4_LINKS, exist_ok=True)
for name in CLIPS:
    src = [p for p in glob.glob(f"{CLIPS_DIR}/*.mp4") if clean(p) == name]
    assert src, f"no video for {name} in {CLIPS_DIR}"
    if not os.path.exists(f"{LOCAL_CLIPS}/{name}.mp4"):
        shutil.copy(src[0], f"{LOCAL_CLIPS}/{name}.mp4")
    ps4 = find_ps4(name)
    assert ps4, f"no PS4 results (hands.jsonl) for {name} anywhere in My Drive"
    need = [f for f in ("hands.jsonl", "camera.json", "head_rot.pkl") if not os.path.exists(f"{ps4}/{f}")]
    assert not need, f"{ps4} is missing {need}"
    link = f"{PS4_LINKS}/{name}"
    if os.path.islink(link): os.remove(link)
    os.symlink(ps4, link)
    mesh = os.path.exists(f"{ps4}/hands_mesh.npz")
    print(f"{name}: video ok | PS4 {ps4} | hand meshes: {'yes' if mesh else 'NO -> skeletons (run the PS4 notebook with MESH_FOR first)'}")

In [ ]:
# CELL 4: SLAM on both whole clips (~7 min each). A finished SLAM saved in Drive is reused after a disconnect.
import subprocess, sys, json, tarfile
LOCAL_SLAM = "/content/ps5_world"; os.makedirs(LOCAL_SLAM, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
SAVE = ["vipe/pose", "vipe/intrinsics", "vipe/step.txt", "vipe/vipe/clip_slam_map.pt", "clip_small.mp4",
        "world_frame.json", "world_hands.jsonl", "world_hands_mesh.npz", "world_stats.json", "trajectory.csv"]

def run(cmd, log):
    print(" ".join(cmd))
    with subprocess.Popen(cmd, cwd="/content/code", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          bufsize=1) as p, open(log, "w") as lf:
        for line in p.stdout:
            print(line, end=""); lf.write(line)
    return p.returncode

for name in CLIPS:
    d, cache = f"{LOCAL_SLAM}/{name}", f"{OUT_DIR}/slam_{name}.tar"
    if os.path.exists(f"{d}/world_frame.json"):
        print(name, ": SLAM already here"); continue
    if os.path.exists(cache):
        with tarfile.open(cache) as tf: tf.extractall(LOCAL_SLAM)
        print(name, ": SLAM restored from Drive"); continue
    rc = run([sys.executable, "run_slam.py", "--clips-dir", LOCAL_CLIPS, "--ps4-dir", PS4_LINKS, "--out-dir", LOCAL_SLAM,
              "--clip", name, "--no-render", "--no-rerun", "--force"], f"{LOCAL_SLAM}/slam_{name}.log")
    assert os.path.exists(f"{d}/world_frame.json"), f"SLAM failed for {name}: see the log above"
    with tarfile.open(cache, "w") as tf:
        for f in SAVE:
            if os.path.exists(f"{d}/{f}"): tf.add(f"{d}/{f}", arcname=f"{name}/{f}")
    st = json.load(open(f"{d}/world_stats.json"))
    print(name, {k: st.get(k) for k in ("backend", "slam_attempts", "hand_meshes", "rotation_1s_vs_ps4_median_deg")})

In [ ]:
# CELL 5: 3D room (~15 min) + 2-min fly-through video (~10-15 min) + web viewer
LOCAL_ROOM = f"/content/room_twin/{TARGET}"
rc = run([sys.executable, "run_room.py", "--clips-dir", LOCAL_CLIPS, "--ps4-dir", PS4_LINKS, "--slam-dir", LOCAL_SLAM,
          "--clips", *CLIPS, "--target", TARGET, "--seconds", str(SECONDS), "--out", LOCAL_ROOM],
         f"/content/room_{TARGET}.log")
assert rc == 0 and os.path.exists(f"{LOCAL_ROOM}/room_video.mp4"), "room step failed: see the log above"
# save to Drive
for f in ("room_mesh.ply", "room.glb", "room_video.mp4", "room_stats.json", "run_room_summary.json"):
    shutil.copy(f"{LOCAL_ROOM}/{f}", f"{OUT_DIR}/{f}")
shutil.make_archive(f"{OUT_DIR}/room_web", "zip", f"{LOCAL_ROOM}/web")
shutil.copy(f"/content/room_{TARGET}.log", f"{OUT_DIR}/run_room.log")
print("saved to", OUT_DIR, sorted(os.listdir(OUT_DIR)))

In [ ]:
# CELL 6: check: registration, scale, a few frames of the video
import json, cv2, numpy as np, matplotlib.pyplot as plt
st = json.load(open(f"{LOCAL_ROOM}/room_stats.json"))
print(json.dumps({k: st[k] for k in ("clips_used", "registration", "depth_scale", "depth_scale_spread", "triangles",
                                     "bbox_min", "bbox_max", "seconds")}, indent=1, default=float))
cap = cv2.VideoCapture(f"{LOCAL_ROOM}/room_video.mp4"); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, ax = plt.subplots(2, 3, figsize=(18, 6.5))
for a, k in zip(ax.ravel(), np.linspace(0, n - 1, 6).astype(int)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k)); ok, fr = cap.read()
    if ok: a.imshow(fr[..., ::-1]); a.set_title(f"t = {k / 30:.0f} s"); a.axis("off")
plt.tight_layout(); plt.show()
# good signs: clips_used has both clips (registration fitness > 0.3, rmse < 20 mm, tilt < 5 deg);
# depth_scale near 1 with spread < 10%; the bench and hands line up in the frames above.

### Open the 3D viewer on your computer
1. Download `room_web.zip` from `MyDrive/mytronD/room_twin/MTL-SKL00089-007/` and unzip it.
2. Double-click `open_viewer.bat`. It starts a small local web server and opens `http://localhost:8000/viewer.html`.
   It works offline: three.js is included.
3. Drag to orbit, right-drag to pan, wheel to zoom. The views are Over shoulder, Worker's eye, Left, Right and Top.
   The slider (or ←/→, Space) moves the hands forward and back.